In [1]:
# Force-reinstall compatible versions (works on Python 3.10-3.12)
%pip install -q --upgrade "gymnasium[atari,accept-rom-license]" "shimmy[atari]" tensorboardX
%pip install ale-py  tensorboard

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import psutil

# Total logical CPUs (including hyperthreading)
print("Logical CPUs:", psutil.cpu_count(logical=True))

# Total physical cores (actual hardware)
print("Physical Cores:", psutil.cpu_count(logical=False))

Logical CPUs: 8
Physical Cores: 8


In [3]:
import sys, os
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir('/content/drive/MyDrive/Colab Notebooks')

if 'google.colab' in sys.modules and not os.path.exists('.setup_complete'):
    # Install xvfb and our launcher script for it
    !apt-get install -y xvfb
    !wget -q https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/xvfb -O ../xvfb

    # Download dependencies from Github
    !wget -q https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/atari_wrappers.py
    !wget -q https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/env_batch.py
    !wget -q https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/runners.py

    # Update the gym environment to be compatible with the Atari environment
    !pip install -q gymnasium[atari,accept-rom-license]
    !pip install -q tensorboardX
    !pip install -q ale_py
    

    !touch .setup_complete

# This code creates a virtual display to draw game images on.
# It will have no effect if your machine has a monitor.
if type(os.environ.get("DISPLAY")) is not str or len(os.environ.get("DISPLAY")) == 0:
    !bash ../xvfb start
    os.environ['DISPLAY'] = ':1'

Starting virtual X frame buffer: Xvfb../xvfb: line 24: start-stop-daemon: command not found
.


# Implementing Advantage-Actor Critic (A2C)

In this notebook you will implement Advantage Actor Critic algorithm that trains on a batch of Atari 2600 environments running in parallel.

Firstly, we will use environment wrappers implemented in file `atari_wrappers.py`. These wrappers preprocess observations (resize, grayscale, take max between frames, skip frames and stack them together) and rewards. Some of the wrappers help to reset the environment and pass `done` flag equal to `True` when agent dies.
File `env_batch.py` includes implementation of `ParallelEnvBatch` class that allows to run multiple environments in parallel. To create an environment we can use `nature_dqn_env` function. Note that if you are using
PyTorch and not using `tensorboardX` you will need to implement a wrapper that will log **raw** total rewards that the *unwrapped* environment returns and redefine the implemention of `nature_dqn_env` function here.



In [4]:
import numpy as np
import gymnasium as gym
from atari_wrappers import nature_dqn_env


env_name = "SpaceInvadersNoFrameskip-v4"
nenvs = 8  # change this if you have more than 8 CPU ;)
summaries = "Tensorboard"

env = nature_dqn_env(env_name, nenvs=nenvs, summaries=summaries)
obs, _ = env.reset()
assert obs.shape == (nenvs, 4, 84, 84)
assert obs.dtype == np.float32


A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


Next, we will need to implement a model that predicts logits and values. It is suggested that you use the same model as in [Nature DQN paper](https://www.nature.com/articles/nature14236) with a modification that instead of having a single output layer, it will have two output layers taking as input the output of the last hidden layer. **Note** that this model is different from the model you used in homework where you implemented DQN. You can use your favorite deep learning framework here. We suggest that you use orthogonal initialization with parameter $\sqrt{2}$ for kernels and initialize biases with zeros.

In [5]:
import torch
import torch.nn as nn
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class NatureCNN(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 512),
            nn.ReLU(),
        )
        self.policy_head = nn.Linear(512, n_actions)
        self.value_head  = nn.Linear(512, 1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.zeros_(m.bias)

    def forward(self, x):
        feat   = self.features(x)
        logits = self.policy_head(feat)
        values = self.value_head(feat).squeeze(-1)
        return logits, values


You will also need to define and use a policy that wraps the model. While the model computes logits for all actions, the policy will sample actions and also compute their log probabilities.  `policy.act` should return a dictionary of all the arrays that are needed to interact with an environment and train the model.
 Note that actions must be an `np.ndarray` while the other
tensors need to have the type determined by your deep learning framework.

In [6]:
class Policy:
    def __init__(self, model):
        self.model = model

    def reset(self):
        pass

    def act(self, inputs):
        obs_t = torch.tensor(np.asarray(inputs), dtype=torch.float32).to(device)
        with torch.no_grad():
            logits, values = self.model(obs_t)
        dist    = torch.distributions.Categorical(logits=logits)
        actions = dist.sample()
        lp      = dist.log_prob(actions)
        return {
            'actions':   actions.cpu().numpy(),
            'logits':    logits,
            'log_probs': lp,
            'values':    values,
        }


Next will pass the environment and policy to a runner that collects partial trajectories from the environment.
The class that does is is already implemented for you.

In [7]:
from runners import EnvRunner

This runner interacts with the environment for a given number of steps and returns a dictionary containing
keys

* 'observations'
* 'rewards'
* 'resets'
* 'actions'
* all other keys that you defined in `Policy`

under each of these keys there is a python `list` of interactions with the environment. This list has length $T$ that is size of partial trajectory. Partial trajectory for given moment `t` is part of `ComputeValueTargets.__call__` input argument `trajectory` from moment `t` to the end (i.e. it's different at each iteration in the algorithm).

To train the part of the model that predicts state values you will need to compute the value targets.
Any callable could be passed to `EnvRunner` to be applied to each partial trajectory after it is collected.
Thus, we can implement and use `ComputeValueTargets` callable.
The formula for the value targets is simple:

$$
\hat v(s_t) = \left( \sum_{t'=0}^{T - 1} \gamma^{t'}r_{t+t'} \right) + \gamma^T \hat{v}(s_{t+T}),
$$

In implementation, however, do not forget to use
`trajectory['resets']` flags to check if you need to add the value targets at the next step when
computing value targets for the current step. You can access `trajectory['state']['latest_observation']`
to get last observations in partial trajectory &mdash; $s_{t+T}$.

In [8]:
class ComputeValueTargets:
    def __init__(self, policy, gamma=0.99):
        self.policy = policy
        self.gamma  = gamma

    def __call__(self, trajectory):
        last_obs  = trajectory['state']['latest_observation']
        last_vals = self.policy.act(last_obs)['values'].cpu().numpy()

        T       = len(trajectory['rewards'])
        targets = [None] * T
        running = last_vals

        for t in reversed(range(T)):
            running    = (trajectory['rewards'][t]
                          + self.gamma * running
                          * (1.0 - trajectory['resets'][t]))
            targets[t] = running

        trajectory['value_targets'] = targets


After computing value targets we will transform lists of interactions into tensors
with the first dimension `batch_size` which is equal to `env_steps * num_envs`, i.e. you essentially need
to flatten the first two dimensions.

In [9]:
class MergeTimeBatch:
    """ Merges first two axes typically representing time and env batch. """
    def __call__(self, trajectory):
        for key, val in list(trajectory.items()):
            if not isinstance(val, list):
                continue
            if isinstance(val[0], np.ndarray):
                stacked = np.stack(val)
                trajectory[key] = stacked.reshape((-1,) + stacked.shape[2:])
            elif torch.is_tensor(val[0]):
                stacked = torch.stack(val)
                trajectory[key] = stacked.reshape((-1,) + stacked.shape[2:])


In [ ]:
n_actions = env.action_space.spaces[0].n
model  = NatureCNN(n_actions).to(device)
policy = Policy(model)
runner = EnvRunner(
    env=env,
    policy=policy,
    nsteps=20,
    transforms=[
        ComputeValueTargets(policy),
        MergeTimeBatch(),
    ],
)


Now is the time to implement the advantage actor critic algorithm itself. You can look into your lecture,
[Mnih et al. 2016](https://arxiv.org/abs/1602.01783) paper, and [lecture](https://www.youtube.com/watch?v=Tol_jw5hWnI&list=PLkFD6_40KJIxJMR-j5A1mkxK26gh_qg37&index=20) by Sergey Levine.

In [ ]:
class A2C:
    def __init__(self,
                 policy,
                 optimizer,
                 value_loss_coef=0.5,
                 entropy_coef=0.02,
                 max_grad_norm=0.5):
        self.policy          = policy
        self.optimizer       = optimizer
        self.value_loss_coef = value_loss_coef
        self.entropy_coef    = entropy_coef
        self.max_grad_norm   = max_grad_norm

    def policy_loss(self, trajectory):
        dist       = torch.distributions.Categorical(logits=trajectory['logits_fresh'])
        log_probs  = dist.log_prob(trajectory['actions_t'])
        entropy    = dist.entropy().mean()
        advantages = (trajectory['value_targets_t']
                      - trajectory['values_fresh'].detach())
        return -(advantages * log_probs).mean(), entropy

    def value_loss(self, trajectory):
        return 0.5 * ((trajectory['value_targets_t']
                       - trajectory['values_fresh']) ** 2).mean()

    def loss(self, trajectory):
        obs          = torch.tensor(trajectory['observations'],
                                    dtype=torch.float32).to(device)
        actions_t    = torch.tensor(trajectory['actions'],
                                    dtype=torch.long).to(device)
        value_tgts_t = torch.tensor(trajectory['value_targets'],
                                    dtype=torch.float32).to(device)

        logits_fresh, values_fresh = self.policy.model(obs)

        aug = dict(trajectory)
        aug['logits_fresh']    = logits_fresh
        aug['values_fresh']    = values_fresh
        aug['actions_t']       = actions_t
        aug['value_targets_t'] = value_tgts_t

        p_loss, entropy = self.policy_loss(aug)
        v_loss          = self.value_loss(aug)
        total           = (p_loss
                           + self.value_loss_coef * v_loss
                           - self.entropy_coef * entropy)
        return total, p_loss, v_loss, entropy

    def step(self, trajectory):
        self.optimizer.zero_grad()
        total, p_loss, v_loss, entropy = self.loss(trajectory)
        total.backward()
        grad_norm = nn.utils.clip_grad_norm_(self.policy.model.parameters(),
                                             self.max_grad_norm)
        self.optimizer.step()
        return total.item(), p_loss.item(), v_loss.item(), entropy.item(), float(grad_norm)


Now you can train your model. With reasonable hyperparameters training on a single GTX1080 for 10 million steps across all batched environments (which translates to about 5 hours of wall clock time)
it should be possible to achieve *average raw reward over last 100 episodes* (the average is taken over 100 last
episodes in each environment in the batch) of about 600. You should plot this quantity with respect to
`runner.step_var` &mdash; the number of interactions with all environments. It is highly
encouraged to also provide plots of the following quantities (these are useful for debugging as well):

* [Coefficient of Determination](https://en.wikipedia.org/wiki/Coefficient_of_determination) between
value targets and value predictions
* Entropy of the policy $\pi$
* Value loss
* Policy loss
* Value targets
* Value predictions
* Gradient norm
* Advantages
* A2C loss

For optimization we suggest you use RMSProp with learning rate starting from 7e-4 and linearly decayed to 0, smoothing constant (alpha in PyTorch and decay in TensorFlow) equal to 0.99 and epsilon equal to 1e-5.

In [ ]:
from tensorboardX import SummaryWriter

TOTAL_STEPS = 15_000_000
NSTEPS      = 20
INIT_LR     = 7e-4
n_updates   = TOTAL_STEPS // (NSTEPS * nenvs)

optimizer = torch.optim.RMSprop(
    model.parameters(), lr=INIT_LR, alpha=0.99, eps=1e-5
)
a2c = A2C(policy, optimizer)

alg_writer = SummaryWriter('logs/a2c_alg')

for update in range(n_updates):
    frac = 1.0 - update / n_updates
    for pg in optimizer.param_groups:
        pg['lr'] = INIT_LR * frac

    trajectory = runner.get_next()
    total_loss, p_loss, v_loss, entropy, grad_norm = a2c.step(trajectory)

    if update % 100 == 0:
        step = runner.step_var

        obs_t = torch.tensor(trajectory['observations'],
                             dtype=torch.float32).to(device)
        with torch.no_grad():
            logits_eval, vp = model(obs_t)
            dist_eval       = torch.distributions.Categorical(logits=logits_eval)
            entropy_eval    = dist_eval.entropy().mean().item()

        vp_np   = vp.cpu().numpy()
        vt_np   = np.asarray(trajectory['value_targets'], dtype=np.float32)
        vt_t    = torch.tensor(vt_np).to(device)
        adv_np  = (vt_t - vp).cpu().numpy()

        ss_res = float(np.sum((vt_np - vp_np) ** 2))
        ss_tot = float(np.sum((vt_np - vt_np.mean()) ** 2))
        r2     = 1.0 - ss_res / (ss_tot + 1e-8)

        alg_writer.add_scalar('Loss/total',              total_loss,           step)
        alg_writer.add_scalar('Loss/policy',             p_loss,               step)
        alg_writer.add_scalar('Loss/value',              v_loss,               step)
        alg_writer.add_scalar('Metrics/entropy',         entropy,              step)
        alg_writer.add_scalar('Metrics/grad_norm',       grad_norm,            step)
        alg_writer.add_scalar('Metrics/r2',              r2,                   step)
        alg_writer.add_scalar('Metrics/value_target_mean', float(vt_np.mean()), step)
        alg_writer.add_scalar('Metrics/value_pred_mean',   float(vp_np.mean()), step)
        alg_writer.add_scalar('Metrics/advantage_mean',    float(adv_np.mean()), step)
        alg_writer.add_scalar('Training/lr',             INIT_LR * frac,       step)

        if update % 1000 == 0:
            print(f'update {update:6d} | step {step:9d} | '
                  f'loss {total_loss:.4f} | p {p_loss:.4f} | v {v_loss:.4f} | '
                  f'ent {entropy:.4f} | r2 {r2:.3f}')

alg_writer.close()
print('Training complete!')


update      0 | step        40 | loss -0.0239 | p -0.0061 | v 0.0000 | ent 1.7773 | r2 -340187.302
update   1000 | step     40040 | loss 0.0148 | p 0.0200 | v 0.0473 | ent 1.7005 | r2 0.897
update   2000 | step     80040 | loss 0.0246 | p 0.0417 | v 0.0025 | ent 1.7704 | r2 0.905
update   3000 | step    120040 | loss 0.2174 | p 0.2185 | v 0.0669 | ent 1.7859 | r2 0.401
update   4000 | step    160040 | loss -0.1062 | p -0.0901 | v 0.0053 | ent 1.7413 | r2 0.975
update   5000 | step    200040 | loss -0.0469 | p -0.0347 | v 0.0156 | ent 1.6132 | r2 0.927
update   6000 | step    240040 | loss -0.0467 | p -0.0327 | v 0.0127 | ent 1.7131 | r2 0.914
update   7000 | step    280040 | loss -0.0893 | p -0.0801 | v 0.0336 | ent 1.7599 | r2 0.815
update   8000 | step    320040 | loss 0.0415 | p 0.0546 | v 0.0165 | ent 1.7184 | r2 0.994
update   9000 | step    360040 | loss -0.0194 | p -0.0040 | v 0.0080 | ent 1.7435 | r2 0.992
update  10000 | step    400040 | loss -0.0651 | p -0.0549 | v 0.0274 | e

Process Process-5:
Process Process-3:
Process Process-1:
Process Process-4:
Process Process-7:
Process Process-8:
Process Process-2:
Process Process-6:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/tsagoll/StudioProjects/ysda_Practical_RL/week06_policy_based/env_batch.py", line 153, in worker
    worker_connection.send((obs, rew, terminated, truncated, info))
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/multipr

KeyboardInterrupt: 

### Target networks?

You may recall a technique called "target networks" we used a few weeks ago when we trained a DQN agent to play Atari Breakout and wonder why we have not suggested using them here. The answer is that this is more historical than practical.

While the "chasing the target" problem is still present in actor-critic value estimation and target networks do show up in follow-up papers, the original A3C/A2C papers do not mention them and do not explain this omission.

The hypothesis why this may not be a big deal (compared to Q-learning) goes like this. An A3C/A2C agent selects actions based on policy, not an epsilon greedy exploration function, for which the argmax can change drastically due to tiny errors in function approximation. Therefore, errors in the value target caused by target chasing will cause less damage.

Also, the actor-critic gradient relies on the advantage function $A(s_t, a_t) = Q(s_t, a_t) - V(s_t)$. Compare this to the $Q$-function $Q(s_t, a_t) = r(s_t, a_t) + \gamma \cdot \mathbb{E}_{s_{t+1} \mid s_t, a_t} V(s_{t+1})$ used in Q-learning and SARSA: we would expect that any bias in $V$-function approximation will be carried over from $V(s_{t+1})$ to $V(s_t)$ by gradient updates. However, in the formula for the advantage function the two approximations ($Q$-function and $V$-function) come with opposite signs, and thus the errors will cancel out.

The last reason may be computational. Authors were concerned to beat existent algorithms in the wall-clock learning time, and any overhead of parameter copying (target network update) counted against this goal.